# 04 | Physics-Informed Feature Engineering

## Study objective

This notebook develops assessment-level predictors that are physically interpretable, available at the time of forecasting and suitable for cross-cell SOH modelling.

The analysis extracts degradation indicators from EIS, IV and transient-response measurements, evaluates feature redundancy and excludes variables that could introduce target or future-information leakage.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sofc_health.models.ml import add_lag_features

ROOT = Path.cwd()

if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

table = pd.read_parquet(ROOT / "data" / "processed" / "modeling_table.parquet")

feature_columns = [column for column in table.columns if column.startswith(("eis_", "iv_", "tr_"))]

print("Modeling-table shape:", table.shape)
print("Number of candidate features:", len(feature_columns))

display(
    table[feature_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_fraction")
    .head(15)
)

## A feature is useful only when it is genuinely available

A feature is not defined only by what it measures. We must also ask when that measurement becomes available.

When forecasting from assessment \(k\), the model may use information recorded at or before that assessment. It must not use the future SOH value that it is trying to predict:

$$
\widehat{\mathrm{SOH}}_{k+h}
=
f\left(\mathbf{x}_{1:k}\right)
$$

Using future measurements, even indirectly, would give the model information that would not exist during real deployment. This is called data leakage.

For a feature \(x\), a valid historical lag is:

$$
x_{k-\ell},
\qquad
\ell \geq 1
$$

For example, \(x_{k-1}\) is the value from the previous assessment, while \(x_{k-2}\) is the value from two assessments earlier.

The same timing rule applies to preprocessing. Missing-value replacement, scaling factors and feature-selection decisions must be learned only from the training data in each validation fold. Otherwise, information from the validation or test period can silently influence the model.

In practical terms, every feature must pass three questions:

1. Is it physically meaningful?
2. Is it available at the time the prediction is made?
3. Can it be calculated without using future information?

A feature that fails any of these checks should not enter the forecasting model.

In [ ]:
candidate = [
    "eis_r_ohmic_ohm",
    "eis_polarization_proxy_ohm",
    "iv_max_power_w_cm2",
    "tr_performance_current_a_cm2",
]
lagged = add_lag_features(table, candidate, lags=(1, 2, 3))
lag_columns = [c for c in lagged if "_lag" in c or c.endswith("_delta1")]
lagged[["cell_id", "assessment_index", *lag_columns]].head(6)

In [ ]:
correlation = table[feature_columns].corr(method="spearman").abs()

upper = correlation.where(
    np.triu(
        np.ones(correlation.shape),
        k=1,
    ).astype(bool)
)

high_pairs = [
    (row, column, upper.loc[row, column])
    for column in upper.columns
    for row in upper.index
    if pd.notna(upper.loc[row, column]) and upper.loc[row, column] > 0.95
]

high_correlation_pairs = pd.DataFrame(
    high_pairs,
    columns=[
        "feature_a",
        "feature_b",
        "abs_spearman",
    ],
).sort_values(
    "abs_spearman",
    ascending=False,
)

high_correlation_pairs.head(20)

In [ ]:
within_cell_features = table[feature_columns] - table.groupby("cell_id")[feature_columns].transform(
    "median"
)

within_correlation = within_cell_features.corr(method="spearman").abs()

within_upper = within_correlation.where(
    np.triu(
        np.ones(within_correlation.shape),
        k=1,
    ).astype(bool)
)

within_high_pairs = [
    (row, column, within_upper.loc[row, column])
    for column in within_upper.columns
    for row in within_upper.index
    if pd.notna(within_upper.loc[row, column]) and within_upper.loc[row, column] > 0.95
]

within_high_correlation_pairs = pd.DataFrame(
    within_high_pairs,
    columns=[
        "feature_a",
        "feature_b",
        "within_cell_abs_spearman",
    ],
).sort_values(
    "within_cell_abs_spearman",
    ascending=False,
)

within_high_correlation_pairs.head(20)

## Conclusion and feature-selection decision

The assessment-level table contains 28 candidate EIS, IV and transient features. Missingness is concentrated in transient features because the final transient assessment is absent for every cell. The \(t_{50}\) and \(t_{90}\) features contain additional missing values when the measured response does not reach the required fraction within the observation window.

Causal lags were generated separately within each cell:

$$
x_{c,k-\ell},
\qquad
\ell \in \{1,2,3\}
$$

The recent change was defined as:

$$
\Delta x_{c,k}
=
x_{c,k}-x_{c,k-1}
$$

Missing values at the beginning of each cell trajectory are structural because earlier assessments do not exist. These values must not be filled using future observations.

Global and within-cell Spearman analysis identified substantial feature redundancy. Some relationships are exact mathematical identities, while others arise because multiple features describe the same IV, EIS or transient response.

The compact model will therefore:

- Retain physically interpretable ohmic and polarization features
- Remove exact mathematical duplicates
- Reduce redundant IV operating-point features
- Retain dynamic transient features that provide distinct information
- Fit imputation and scaling only inside each training fold
- Compare feature groups through ablation on held-out cells

Feature selection will not be based only on one correlation threshold or one importance ranking. A feature will be retained when it is physically meaningful, available at prediction time, non-leaking and consistently useful across unseen cells.